# Notebook 4 - CNNs in practice

**UKACM Autumn School: AI for Computational Mechanics**

### Where Notebook 3 left off

Notebook 3 built a convolutional network, trained it on the microstructures, showed with a
permutation test that it genuinely uses the spatial arrangement, and then reported an uncomfortable
result: on the anisotropy $\Delta E$ it scored below a gradient boosting model fitted to 14
hand-built descriptors.

That is the end of "does it train". This notebook is about everything between "it trains" and
"I would put my name on this number".

### What you will do

Five questions, in the order you will meet them on your own problem.

1. **Part 1.** How much data do you actually need? A learning curve says whether you are
   data-limited or model-limited, and it is the first thing to measure and the thing most often
   skipped.
2. **Part 2.** Augmentation as physics. Which transformations are exact symmetries of this problem,
   which are symmetries only if the target is transformed too, and what it costs to get that wrong.
3. **Part 3.** Physical bounds. What Voigt and Reuss do and do not bound, how to build an envelope
   bound into the network output, and whether it helped here.
4. **Part 4.** Where it fails. Train inside volume fraction 10 to 40 percent, predict at 50 and 60,
   and look at the damage.
5. **Part 5.** Uncertainty on a budget. A small ensemble, and whether its spread actually tracks
   the error.

### Table of contents

| Part | Question |
|---|---|
| 1 | Am I data-limited or model-limited? |
| 2 | Which augmentations are allowed by the physics? |
| 3 | Can a known bound be built into the network? |
| 4 | What happens outside the training range? |
| 5 | Does the model know when it does not know? |


In [ ]:
# --- Setup -------------------------------------------------------------------
# Run this first.

import os, io, time, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox

import torch
import torch.nn as nn
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)          # matches a free Colab CPU runtime

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.grid": True,
    "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False,
    "animation.embed_limit": 60,
})

C_DATA, C_FIT, C_ALT, C_BAD = "#3B6EA5", "#C25E00", "#4C9A5E", "#A8323E"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("numpy", np.__version__, "| torch", torch.__version__, "| device", DEVICE)
if DEVICE == "cpu":
    print("No GPU found. Every timing quoted below is for 2 CPU cores.")


### Loading the data

The next cell downloads the labels, images and fourteen-descriptor dataset automatically.


In [ ]:
# --- Load data ---------------------------------------------------------------
DATA_URL = "https://raw.githubusercontent.com/CEMS-Lab/autumn-school/main/datasets/machine_learning/NB4_data.zip"
DATA_DIR = "."

FILES = ["microstructure_labels.csv", "microstructures_64.npz",
         "microstructure_descriptors_core14_64.npz"]

def _have():
    return all(os.path.exists(os.path.join(DATA_DIR, f)) for f in FILES)

if not _have() and DATA_URL:
    print("Downloading ...")
    with urllib.request.urlopen(DATA_URL) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(DATA_DIR)

if not _have():
    try:
        from google.colab import files
        print("Select:", ", ".join(FILES))
        files.upload()
    except ImportError:
        raise FileNotFoundError(
            "Put " + ", ".join(FILES) + " beside the notebook, or set DATA_URL above.")

df = pd.read_csv(os.path.join(DATA_DIR, "microstructure_labels.csv"))
df["E_mean"] = (df.E22 + df.E33) / 2
df["dE"]     =  df.E22 - df.E33
df["vf"]     =  df.vol_frac / 100.0

clean = df[~df.outlier_flag].copy().reset_index(drop=True)

_img = np.load(os.path.join(DATA_DIR, "microstructures_64.npz"), allow_pickle=True)
IMAGES, IMG_KEYS = _img["images"], list(_img["keys"])
IMG_IDX = {k: i for i, k in enumerate(IMG_KEYS)}

_dsc = np.load(os.path.join(DATA_DIR, "microstructure_descriptors_core14_64.npz"), allow_pickle=True)
DESC_ALL, DESC_KEYS, DESC_NAMES = _dsc["D"], list(_dsc["keys"]), list(_dsc["names"])
DESC_IDX = {k: i for i, k in enumerate(DESC_KEYS)}

X_IMG  = IMAGES[[IMG_IDX[k]  for k in clean.key]].astype(np.float32)
X_DESC = DESC_ALL[[DESC_IDX[k] for k in clean.key]].astype(np.float32)
y_mean = clean["E_mean"].values.astype(np.float32)
y_dE   = clean["dE"].values.astype(np.float32)
y_E22  = clean["E22"].values.astype(np.float32)
y_E33  = clean["E33"].values.astype(np.float32)
VF     = clean["vf"].values.astype(np.float32)

idx = np.arange(len(clean))
itr, ite = train_test_split(idx, test_size=0.25, random_state=SEED)

print(f"{len(clean)} microstructures kept, {int(df.outlier_flag.sum())} outliers dropped")
print(f"train {len(itr)}   test {len(ite)}   images {X_IMG.shape}   descriptors {X_DESC.shape}")


### The network and the training loop

The architecture is the one from Notebook 3, unchanged: three blocks of 3x3 convolution, ReLU and
2x2 max pooling with circular padding, then a small dense head. Keeping it fixed is the point. This
notebook varies the data, the augmentation and the output layer, not the architecture.

One change to the training helper matters. `augment` now returns a transformed **target** as well as
a transformed image, because Part 2 needs a transformation that acts on both.

In [ ]:
# --- Architecture and augmentation -------------------------------------------
def make_cnn(widths=(16, 32, 64), head=64, in_size=64, pad_mode="circular", seed=SEED):
    torch.manual_seed(seed)
    layers, c = [], 1
    for w in widths:
        layers += [nn.Conv2d(c, w, 3, padding=1, padding_mode=pad_mode),
                   nn.ReLU(), nn.MaxPool2d(2)]
        c = w
    s = in_size // (2 ** len(widths))
    layers += [nn.Flatten(), nn.Linear(c * s * s, head), nn.ReLU(), nn.Linear(head, 1)]
    return nn.Sequential(*layers)


def augment(xb, yb, mode="roll+flip", rng=np.random, target_parity="even", y_off=0.0):
    # Every transformation is drawn INDEPENDENTLY PER IMAGE, not once per batch. Drawing
    # one shift for a whole batch is a common shortcut and it wastes most of the effect.
    #
    # target_parity says how the target behaves when the two in-plane axes are exchanged:
    #   'even'  E_mean is unchanged by a 90 degree rotation
    #   'odd'   dE = E22 - E33 changes sign
    # yb arrives standardised, yb = (y - mu)/sd, so the odd target is negated in physical
    # units: (-y - mu)/sd = -yb - 2*mu/sd. y_off carries that 2*mu/sd.
    if mode == "none":
        return xb, yb
    B, C, H, W = xb.shape
    dev = xb.device

    sy = torch.as_tensor(rng.randint(0, H, size=B), device=dev)
    sx = torch.as_tensor(rng.randint(0, W, size=B), device=dev)
    iy = ((torch.arange(H, device=dev).view(1, H) - sy.view(B, 1)) % H)
    xb = torch.gather(xb, 2, iy.view(B, 1, H, 1).expand(B, C, H, W))
    ix = ((torch.arange(W, device=dev).view(1, W) - sx.view(B, 1)) % W)
    xb = torch.gather(xb, 3, ix.view(B, 1, 1, W).expand(B, C, H, W))
    if mode == "roll":
        return xb, yb

    mx = torch.as_tensor(rng.rand(B) < 0.5, device=dev).view(B, 1, 1, 1)
    xb = torch.where(mx, torch.flip(xb, dims=[3]), xb)                 # mirror in x
    my = torch.as_tensor(rng.rand(B) < 0.5, device=dev).view(B, 1, 1, 1)
    xb = torch.where(my, torch.flip(xb, dims=[2]), xb)                 # mirror in y
    if mode == "roll+flip":
        return xb, yb

    rt = torch.as_tensor(rng.rand(B) < 0.5, device=dev)                # 90 degree rotation
    xb = torch.where(rt.view(B, 1, 1, 1), torch.rot90(xb, 1, dims=(2, 3)), xb)
    if mode == "roll+flip+rot" and target_parity == "odd":             # transform the target too
        yb = torch.where(rt.view(B, 1), -yb - y_off, yb)
    # mode == "roll+flip+rot_wrong" deliberately leaves the target alone
    return xb, yb


N_PARAM = sum(p.numel() for p in make_cnn().parameters())
print(f"CNN parameters: {N_PARAM:,d}")
print("augmentation modes:", ["none", "roll", "roll+flip", "roll+flip+rot", "roll+flip+rot_wrong"])


In [ ]:
# --- Training helper ----------------------------------------------------------
def train_cnn(model, Xtr, ytr, Xte, yte, epochs=20, bs=32, lr=1e-3, aug="roll+flip",
              target_parity="even", standardise=True, eval_every=0, seed=SEED, tag="",
              quiet=False):
    rng = np.random.RandomState(seed)
    torch.manual_seed(seed)
    model = model.to(DEVICE)

    mu, sd = (float(ytr.mean()), float(ytr.std())) if standardise else (0.0, 1.0)
    A = torch.tensor(Xtr).unsqueeze(1).to(DEVICE)
    B = torch.tensor((ytr - mu) / sd).view(-1, 1).to(DEVICE)
    P = torch.tensor(Xte).unsqueeze(1).to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.MSELoss()
    n = len(A)
    hist_ep, hist_r2, hist_loss = [], [], []

    t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        run = 0.0
        for i in range(0, n, bs):
            j  = perm[i:i + bs]
            xb, yb = augment(A[j].clone(), B[j], aug, rng, target_parity,
                             y_off=2.0 * mu / sd)
            opt.zero_grad()
            l = lossf(model(xb), yb)
            l.backward(); opt.step()
            run += l.item() * len(j)
        hist_loss.append(run / n)
        if eval_every and (ep % eval_every == eval_every - 1 or ep == epochs - 1):
            model.eval()
            with torch.no_grad():
                pr = model(P).cpu().numpy().ravel() * sd + mu
            hist_ep.append(ep + 1); hist_r2.append(r2_score(yte, pr))
    elapsed = time.time() - t0

    model.eval()
    with torch.no_grad():
        pred = model(P).cpu().numpy().ravel() * sd + mu
    r2 = r2_score(yte, pred)
    if not quiet:
        print(f"{tag}n_train={n:4d}  {epochs} epochs  aug={aug:<20s} "
              f"{elapsed:6.1f} s ({elapsed/epochs:.2f} s/epoch)  test R2 = {r2:+.4f}")
    return dict(r2=r2, elapsed=elapsed, pred=pred, ep=hist_ep, r2_hist=hist_r2,
                loss=hist_loss, model=model)


def predict(model, X, mu=0.0, sd=1.0, bs=256):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            xb = torch.tensor(X[i:i+bs]).unsqueeze(1).to(DEVICE)
            out.append(model(xb).cpu().numpy().ravel())
    return np.concatenate(out) * sd + mu


### How accuracy is reported

One number runs through all six notebooks and it has not yet been written down, so here it is once.
For $N$ test microstructures with true values $y_i$, predictions $\hat{y}_i$ and mean
$\bar{y} = \frac{1}{N}\sum_i y_i$, the coefficient of determination is

$$\boxed{\;R^2 \;=\; 1 \;-\; \frac{\sum_{i=1}^{N}\left(y_i - \hat{y}_i\right)^2}
{\sum_{i=1}^{N}\left(y_i - \bar{y}\right)^2}\;}$$

The numerator is the squared error the model leaves behind, the denominator is the squared error of
predicting the mean of the test set for everything. So $R^2$ is dimensionless, $R^2 = 1$ is a perfect
fit, $R^2 = 0$ means the model is no better than the mean, and $R^2 < 0$ means it is worse. Negative
values appear several times below and they are not a bug.

The other number used here is the root mean squared error,
$\mathrm{RMSE} = \bigl(\frac{1}{N}\sum_i (y_i - \hat{y}_i)^2\bigr)^{1/2}$, which keeps the units of
the target, GPa. $R^2$ says how much of the variation is captured, RMSE says how wrong a single
prediction typically is.

---

# Part 1 - How much data do you actually need

Before tuning anything, find out which wall you are against.

- If the test score is still climbing steeply as the last images are added, you are **data-limited**.
  More FE runs will buy more accuracy, and no amount of architecture work will substitute.
- If the curve has flattened, you are **model-limited**. More data buys nothing and the effort
  belongs in the representation, the architecture or the loss.

The measurement is simple: train the same model on nested subsets of the training set and plot the
test score against the training set size. The test set is held fixed throughout, so the numbers are
comparable.

One thing has to be done properly or the whole measurement is worthless. **Each training size is run
with three different random seeds**, and what is plotted is the mean with the individual runs shown
as well. A single run per point is not enough here, and the spread printed below is the reason.

The target is $\Delta E$, the anisotropy, because that is the hard one. Each run uses a short budget
of epochs so twelve runs fit in the lecture. The absolute scores are therefore lower than the longer
runs in Notebook 3, and only the shape of the curve is being read.

In [ ]:
# --- Learning curve on dE, three seeds per training size ----------------------
FRACTIONS  = [0.10, 0.25, 0.50, 1.00]
SEEDS_LC   = [0, 1, 2]
EPOCHS_LC  = 12 if DEVICE == "cpu" else 30      # short, so three seeds per point fit

rng_lc = np.random.RandomState(SEED)
order  = rng_lc.permutation(len(itr))           # nested subsets: each contains the previous
Xte_img, yd_te, ym_te = X_IMG[ite], y_dE[ite], y_mean[ite]

lc_n, lc_all, lc_pred, lc_r2_s0 = [], [], [], []
t_all = time.time()
for f in FRACTIONS:
    k   = max(16, int(round(f * len(itr))))
    sub = itr[order[:k]]
    row = []
    for sd_ in SEEDS_LC:
        r = train_cnn(make_cnn(seed=sd_), X_IMG[sub], y_dE[sub], Xte_img, yd_te,
                      epochs=EPOCHS_LC, aug="roll+flip", target_parity="odd", seed=sd_,
                      tag=f"{int(f*100):>3d}% of train, seed {sd_}: ")
        row.append(r["r2"])
        if sd_ == SEEDS_LC[0]:
            lc_pred.append(r["pred"]); lc_r2_s0.append(r["r2"])
    lc_n.append(k); lc_all.append(row)
lc_all = np.array(lc_all)                       # (n_sizes, n_seeds)
lc_r2  = lc_all.mean(axis=1)
print(f"\nwhole sweep: {time.time() - t_all:.1f} s "
      f"({len(FRACTIONS)}x{len(SEEDS_LC)} runs)")


In [ ]:
# --- The learning curve, with the seed spread shown ---------------------------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))

lo_s, hi_s = lc_all.min(axis=1), lc_all.max(axis=1)
a1.fill_between(lc_n, lo_s, hi_s, color=C_DATA, alpha=0.18, label="range over 3 seeds")
a1.plot(lc_n, lc_r2, "-o", color=C_DATA, lw=2, ms=6, label="mean of 3 seeds")
for j, sd_ in enumerate(SEEDS_LC):
    a1.plot(lc_n, lc_all[:, j], "x", color=C_BAD, ms=7, mew=1.6,
            label="individual runs" if j == 0 else None)
a1.set_xlabel("training microstructures"); a1.set_ylabel("test $R^2$ on $\\Delta E$")
a1.set_title(f"learning curve, {EPOCHS_LC} epochs, 3 seeds per point", fontsize=10)
a1.set_ylim(min(-0.15, lc_all.min() - 0.1), max(0.9, lc_all.max() + 0.15))
a1.axhline(0, color="k", lw=0.8)
a1.legend(fontsize=7.5, loc="upper left")

# how the seed-to-seed spread compares with the size-to-size change
a2.bar(np.arange(len(lc_n)) - 0.18, hi_s - lo_s, 0.36, color=C_BAD,
       label="spread across seeds, same $n$")
step = np.abs(np.diff(np.concatenate([[lc_r2[0]], lc_r2])))
a2.bar(np.arange(len(lc_n)) + 0.18, step, 0.36, color=C_ALT,
       label="change in the mean from the previous $n$")
a2.set_xticks(np.arange(len(lc_n))); a2.set_xticklabels([str(n_) for n_ in lc_n])
a2.set_xlabel("training microstructures"); a2.set_ylabel("$R^2$")
a2.set_title("is the trend bigger than the noise?", fontsize=10)
a2.legend(fontsize=7.5)
plt.tight_layout(); plt.show()

print(f"{'n_train':>8s} {'mean R2':>9s} {'min':>8s} {'max':>8s} {'spread':>8s}")
for i, n_ in enumerate(lc_n):
    print(f"{n_:>8d} {lc_r2[i]:>9.4f} {lo_s[i]:>8.4f} {hi_s[i]:>8.4f} {hi_s[i]-lo_s[i]:>8.4f}")
print(f"\nchange in the mean over the last doubling ({lc_n[-2]} to {lc_n[-1]} images): "
      f"{lc_r2[-1] - lc_r2[-2]:+.4f} in R2")
print(f"largest seed-to-seed spread at a single training size:         "
      f"{float((hi_s - lo_s).max()):+.4f} in R2")
print(f"number of runs that collapsed to R2 < 0.05 (no better than the mean): "
      f"{int((lc_all < 0.05).sum())} of {lc_all.size}")


### Reading it

Read the right-hand panel before the left one. It puts the seed-to-seed spread at a fixed training
size next to the change in the mean from one size to the next. Where the orange bar is not clearly
taller than the red one, the curve is not telling you anything about data.

That is the first lesson of this part, and it is not the one the section was set up to teach. At this
training set size and this epoch budget, the run-to-run variation from the random seed alone is
comparable with the effect being measured. The printed spread is how large it is, and the printed
count of collapsed runs is how bad it gets: some runs of this architecture land at $R^2$ near zero,
meaning the network ended up predicting roughly the training mean for everything.

So the honest reading is conditional. If the change over the last doubling is clearly larger than
the spread, the curve has not flattened and more FE runs are the cheapest improvement available. If
it is not, the measurement is not yet precise enough to say, and the fix is more seeds or a longer
budget, not a deeper network.

Two further warnings that apply whatever the numbers do.

The curve depends on the training budget. Each point gets the same short epoch count, so the large
subsets are trained less thoroughly per image than the small ones. A curve measured at a fixed
number of gradient steps and a curve measured at a fixed number of epochs answer different questions.

Three seeds is the minimum that shows a spread at all. Published learning curves normally use five
or more, with the standard error drawn. A learning curve from one run per point, which is what you
will most often see, cannot be distinguished from this one.

### The animation: the same scatter, filled in with data

The prediction scatter at each training size, using the first seed at each point. A model trained on
too little data does not just scatter more, it also regresses towards the mean, which shows up as a
best fit line flatter than the diagonal. The fitted slope is printed in each frame, and 1.00 is what
an unbiased model would give. Read the slopes against the $R^2$ values in the right panel rather
than expecting a clean progression: these are single runs, and the previous cell measured how much
single runs move.

In [ ]:
# --- Animation: the scatter tightening as training data is added -------------
PER    = 12                                   # frames held at each training size
N_FR   = PER * len(FRACTIONS)
srt    = np.argsort(yd_te)                    # reveal points left to right, not at random
slopes = [float(np.polyfit(yd_te, p, 1)[0]) for p in lc_pred]

# Limits must contain the predictions as well as the truth, otherwise the early
# stages, which regress hard towards the mean, sit on an axis chosen for data they
# never reach and the panel looks empty for the wrong reason.
allv = np.concatenate([yd_te] + [np.asarray(p) for p in lc_pred])
pad  = 0.08 * (allv.max() - allv.min())
lim  = [float(allv.min() - pad), float(allv.max() + pad)]

fig, (axA, axB2) = plt.subplots(1, 2, figsize=(10.6, 4.6),
                                gridspec_kw={"width_ratios": [1.05, 1]})

axA.plot(lim, lim, "k--", lw=1, zorder=1)
sc = axA.scatter(yd_te, lc_pred[0], s=14, alpha=0.55,
                 facecolor=C_DATA, edgecolor="none", zorder=3)
line, = axA.plot([], [], color=C_FIT, lw=2, zorder=4)
axA.set_xlim(lim); axA.set_ylim(lim)
axA.set_xlabel("true $\\Delta E$ (GPa)"); axA.set_ylabel("predicted $\\Delta E$ (GPa)")
ttl = axA.set_title("", fontsize=10)

axB2.plot(lc_n, lc_r2, "-", color="0.8", lw=1.5, zorder=1)
(lcline,) = axB2.plot([], [], "-o", color=C_DATA, lw=2, ms=6, zorder=3)
(lchead,) = axB2.plot([], [], "o", color=C_FIT, ms=11, mec="w", mew=1.5, zorder=4)
axB2.axhline(0, color="k", lw=0.8)
axB2.set_xlabel("training microstructures"); axB2.set_ylabel("test $R^2$ on $\\Delta E$")
axB2.set_xlim(0, max(lc_n) * 1.12)
axB2.set_ylim(min(-0.12, min(lc_r2) - 0.1), max(0.6, max(lc_r2) + 0.15))
axB2.set_title("the learning curve, filling in", fontsize=10)

def update_lc(f):
    s = min(f // PER, len(FRACTIONS) - 1)
    k = int(round((((f % PER) + 1) / PER) * len(yd_te)))
    k = max(6, min(k, len(yd_te)))
    sel = srt[:k]
    p = lc_pred[s]
    sc.set_offsets(np.c_[yd_te[sel], np.asarray(p)[sel]])
    line.set_data(lim, np.polyval(np.polyfit(yd_te, p, 1), lim))
    ttl.set_text(f"{lc_n[s]} training images, seed {SEEDS_LC[0]}, "
                 f"test $R^2$ = {lc_r2_s0[s]:+.3f}\n"
                 f"best fit slope {slopes[s]:.2f} (1.00 would be unbiased)")
    lcline.set_data(lc_n[:s+1], lc_r2[:s+1])
    lchead.set_data([lc_n[s]], [lc_r2[s]])
    return sc, line, lcline, lchead, ttl

anim_lc = animation.FuncAnimation(fig, update_lc, frames=N_FR, interval=110, blit=False)
plt.close(fig)
HTML(anim_lc.to_jshtml())


**What the animation shows.** Left: the test predictions against the truth at each training
set size for the first seed, with the dashed diagonal an unbiased model would lie on and the orange
line the best fit through the cloud. Right: the mean learning curve, filling in as the left panel
advances.

Watch the orange line rather than the scatter width. A run that has learned little returns a nearly
horizontal line, which is a model predicting close to the training mean for every input. The printed
slope is how far short of the diagonal it falls, and a slope well below one is the signature of a
model with too little data, or of a run that failed, rather than of a noisy target.

The vertical spread is the other half of the picture. Both effects are real, and only the second is
what people usually mean by "not enough data". The slopes here come from one seed each, so compare
them with the spread measured in the previous cell before reading a trend into them.

---

# Part 2 - Augmentation is physics, not a trick

Augmentation multiplies the training set by applying transformations that leave the target
unchanged. In computer vision the list is conventional: crops, flips, small rotations, colour
jitter. In computational mechanics the list is not conventional, it is derived, and the derivation
is the interesting part.

### The condition, written down

Let $I$ be a microstructure image, $T$ a transformation acting on it, and $y(\cdot)$ the property
the network is being trained to predict. Augmenting the training set with the pair $(T I,\, y(I))$
is correct if and only if

$$\boxed{\;y(T\,I) \;=\; y(I)\;}$$

that is, the transformation is a symmetry of the target, not merely a plausible edit of the picture.
When the transformation acts on the target in a known way rather than leaving it alone, the weaker
condition is enough,

$$y(T\,I) \;=\; S_T\,y(I)$$

with $S_T$ the known action of $T$ on the target, and the augmented pair is then
$(T I,\, S_T\, y(I))$. Everything in this part is working out $S_T$ for each candidate.

Take $y$ to be the pair $(E_{22}, E_{33})$ in GPa, from which $E_{mean}$ and $\Delta E$ follow. The
candidates, with $R_{s}$ a cyclic shift by $s$ pixels, $M_x$, $M_y$ the mirrors and
$R_{90}$ the quarter turn:

| $T$ | $S_T$ acting on $(E_{22}, E_{33})$ | Valid with the label unchanged? |
|---|---|---|
| $R_{s_x, s_y}$, periodic roll | identity | yes, exactly |
| $M_x$, mirror in $x$ | identity | yes |
| $M_y$, mirror in $y$ | identity | yes |
| $R_{90}$, rotate 90 degrees | $(E_{22}, E_{33}) \mapsto (E_{33}, E_{22})$ | **no**, only with the targets swapped |
| crop and tile | not known | no, it is not the same material cell |

For the two derived targets this reads
$E_{mean}(R_{90} I) = E_{mean}(I)$ and $\Delta E(R_{90} I) = -\Delta E(I)$: the mean is even under
exchange of the in-plane axes, the anisotropy is odd. That sign is the whole of Part 2.

A transformation is admissible if it maps a valid microstructure to a valid microstructure and if
you know exactly what it does to the target. Three cases arise here.

**Periodic roll, an exact symmetry.** The cells are periodic. Shifting the image cyclically in $x$
or $y$ is the same physical cell described with a different origin, so every homogenised property is
identical. `torch.roll` with wraparound is exactly this operation. It costs nothing and there are
$64 \times 64 = 4096$ distinct shifts of every image.

**Mirror in $x$ or in $y$, a symmetry of the target.** Reflection maps the cell to another admissible
periodic cell, and it does not exchange the two in-plane axes, so $E_{22}$ and $E_{33}$ are each
preserved. Both $E_{mean}$ and $\Delta E$ survive unchanged.

**Rotation by 90 degrees, a symmetry only if the target is transformed too.** This one exchanges the
$x$ and $y$ axes. The rotated cell is admissible, but its $E_{22}$ is the original's $E_{33}$.
Therefore $E_{mean}$ is unchanged and $\Delta E$ changes sign. Applied with the target left alone it
is not augmentation, it is mislabelling.

That last case is the one worth being careful about, so it is measured below rather than asserted.

### Checking the symmetries on a descriptor, not on faith

There is no FE solver in this notebook, so the invariance of $E_{22}$ and $E_{33}$ cannot be
recomputed directly. It can be checked on the statistic that Notebook 2 found carries the
anisotropy: `S2diff`, the difference between the periodic two-point correlation along $x$ and along
$y$ at a fixed lag.

$$S_2^{x}(h) = \frac{1}{N}\sum_{i,j} I(i,j)\, I(i,\, j+h \bmod n), \qquad
  S_2^{y}(h) = \frac{1}{N}\sum_{i,j} I(i,j)\, I(i+h \bmod n,\, j)$$

The two-point correlation is computed with wraparound, so it inherits the periodicity of the cell.
If a transformation leaves $S_2^{x} - S_2^{y}$ unchanged, it is consistent with leaving $\Delta E$
unchanged. If it flips the sign, the target must flip too.

In [ ]:
# --- Periodic two-point correlation, and how it behaves under each transform --
def S2_xy(img, h=12):
    Ix = np.roll(img, -h, axis=1)      # lag h along x, periodic
    Iy = np.roll(img, -h, axis=0)      # lag h along y, periodic
    return float((img * Ix).mean()), float((img * Iy).mean())

TRANSFORMS = {
    "original":            lambda a: a,
    "roll x by 17":        lambda a: np.roll(a, 17, axis=1),
    "roll y by 9":         lambda a: np.roll(a, 9, axis=0),
    "roll x and y":        lambda a: np.roll(np.roll(a, 23, axis=1), 31, axis=0),
    "mirror in x":         lambda a: a[:, ::-1],
    "mirror in y":         lambda a: a[::-1, :],
    "rotate 90 degrees":   lambda a: np.rot90(a, 1),
    "rotate 180 degrees":  lambda a: np.rot90(a, 2),
    "crop centre 48, tile":lambda a: np.tile(a[8:56, 8:56], (2, 2))[:64, :64],
}

i0 = int(np.argmax(np.abs(y_dE)))                 # a strongly anisotropic example
img0 = X_IMG[i0]
s2x0, s2y0 = S2_xy(img0)

print(f"microstructure {clean.key[i0]}   E22 = {y_E22[i0]:.3f}   E33 = {y_E33[i0]:.3f}   "
      f"dE = {y_dE[i0]:+.3f} GPa\n")
print(f"{'transform':22s} {'Vf':>7s} {'S2x':>8s} {'S2y':>8s} {'S2x-S2y':>9s} {'verdict':>28s}")
for name, fn in TRANSFORMS.items():
    a = np.ascontiguousarray(fn(img0))
    sx, sy = S2_xy(a)
    d = sx - sy
    same_vf = abs(a.mean() - img0.mean()) < 1e-6
    if same_vf and abs(d - (s2x0 - s2y0)) < 1e-6:
        v = "target unchanged"
    elif same_vf and abs(d + (s2x0 - s2y0)) < 1e-6:
        v = "axes swapped, dE changes sign"
    else:
        v = "NOT a symmetry"
    print(f"{name:22s} {a.mean():7.4f} {sx:8.4f} {sy:8.4f} {d:+9.4f} {v:>28s}")


The rolls and the mirrors leave $S_2^{x} - S_2^{y}$ alone. The 90 degree rotation flips its
sign, exactly as the argument about exchanging the axes predicts, and the 180 degree rotation is a
pair of mirrors so it leaves everything alone.

The last row is there as a warning. Cropping the centre of the cell and tiling it produces something
that looks like a microstructure, and it is a perfectly reasonable augmentation for a photograph. It
is not a periodic cell of this material. The volume fraction has moved, fibres are cut at the crop
boundary, and the homogenised stiffness of the thing in the picture is not the label attached to it.

### The same argument as a picture

One microstructure under each candidate transformation, with the label that would be attached to it
and a mark saying whether the original label still applies.

In [ ]:
# --- Schematic: is the target preserved by each transformation? --------------
CAND = [("original",        r"$I$",                   lambda a: a,                      "keep"),
        ("periodic roll",   r"$R_{17,9}\,I$",
         lambda a: np.roll(np.roll(a, 17, axis=1), 9, axis=0),                          "keep"),
        ("mirror in $x$",   r"$M_x\,I$",              lambda a: a[:, ::-1],             "keep"),
        ("mirror in $y$",   r"$M_y\,I$",              lambda a: a[::-1, :],             "keep"),
        ("rotate 90 deg",   r"$R_{90}\,I$",           lambda a: np.rot90(a, 1),         "swap"),
        ("crop 48, tile",   r"$C\,I$",
         lambda a: np.tile(a[8:56, 8:56], (2, 2))[:64, :64],                            "break")]

E22_0, E33_0 = float(y_E22[i0]), float(y_E33[i0])

def _tick(ax, x, y, col, s=0.055):
    ax.plot([x - s, x - s*0.15, x + s*1.05], [y, y - s*0.9, y + s*0.95],
            color=col, lw=2.6, solid_capstyle="round",
            transform=ax.transAxes, clip_on=False, zorder=9)

def _cross(ax, x, y, col, s=0.05):
    ax.plot([x - s, x + s], [y - s, y + s], color=col, lw=2.6, solid_capstyle="round",
            transform=ax.transAxes, clip_on=False, zorder=9)
    ax.plot([x - s, x + s], [y + s, y - s], color=col, lw=2.6, solid_capstyle="round",
            transform=ax.transAxes, clip_on=False, zorder=9)

fig, axes = plt.subplots(1, len(CAND), figsize=(14.5, 4.8))
# The verdict text and the tick or cross sit below each image in axes coordinates,
# so the bottom margin is set by hand: tight_layout would not leave room for them.
fig.subplots_adjust(left=0.02, right=0.98, top=0.80, bottom=0.34, wspace=0.12)
for ax, (name, sym, fn, kind) in zip(axes, CAND):
    a = np.ascontiguousarray(fn(img0))
    ax.imshow(a, cmap="gray", interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title(f"{name}\n{sym}", fontsize=9.5)

    if kind == "keep":
        e22, e33 = E22_0, E33_0
        lab = "label reused as it is"; col = C_ALT
    elif kind == "swap":
        e22, e33 = E33_0, E22_0
        lab = "labels must be swapped"; col = C_BAD
    else:
        e22 = e33 = np.nan
        lab = "no label applies"; col = C_BAD

    if np.isnan(e22):
        body = "$E_{22}$ = ?    $E_{33}$ = ?\n$\\Delta E$ = ?"
    else:
        body = (f"$E_{{22}}$ = {e22:.2f}    $E_{{33}}$ = {e33:.2f}\n"
                f"$\\Delta E$ = {e22 - e33:+.2f} GPa")
    ax.text(0.5, -0.09, body, transform=ax.transAxes, ha="center", va="top", fontsize=8.5)
    ax.text(0.5, -0.30, lab, transform=ax.transAxes, ha="center", va="top",
            fontsize=8.5, color=col)
    if kind == "keep":
        _tick(ax, 0.5, -0.48, C_ALT)
    else:
        _cross(ax, 0.5, -0.48, C_BAD)

fig.suptitle("Does the original label survive the transformation?  "
             "tick: yes.  cross: no, the label has to change or cannot be assigned.",
             fontsize=10, y=0.97)
plt.show()


**What the schematic shows.** Six versions of the same cell. Under each is the label that
would be attached to it in an augmented training set, and a mark saying whether that is the original
label. The first four carry a tick: the picture has changed, the pair $(E_{22}, E_{33})$ has not, so
each one is a free extra training example.

The fifth carries a cross even though the image is a perfectly good microstructure. The quarter turn
exchanges the in-plane axes, so the two moduli swap and $\Delta E$ changes sign. Used with the
original label it teaches the network that the same cell has two opposite anisotropies.

The sixth carries a cross for a different reason. Cropping and tiling makes a picture that no longer
corresponds to the simulation that produced the label at all. No swap rescues it.

### Interactive: which transformations leave the target alone

Pick a microstructure and a transformation. The panel shows the original and the transformed cell,
with the target that the transformed image should be trained against.

In [ ]:
# --- Interactive transformation explorer -------------------------------------
def transform_explorer(sample=i0, transform="rotate 90 degrees", show_S2=True):
    img = X_IMG[sample]
    a = np.ascontiguousarray(TRANSFORMS[transform](img))
    E22_o, E33_o = float(y_E22[sample]), float(y_E33[sample])

    sx0, sy0 = S2_xy(img); sx1, sy1 = S2_xy(a)
    same_vf  = abs(a.mean() - img.mean()) < 1e-6
    swapped  = same_vf and abs((sx1 - sy1) + (sx0 - sy0)) < 1e-6 and abs(sx0 - sy0) > 1e-6
    kept     = same_vf and abs((sx1 - sy1) - (sx0 - sy0)) < 1e-6

    if kept:
        E22_n, E33_n, verdict = E22_o, E33_o, "exact symmetry: reuse the label as it is"
    elif swapped or transform.startswith("rotate 90"):
        E22_n, E33_n, verdict = E33_o, E22_o, "axes exchanged: swap the label, dE changes sign"
    else:
        E22_n = E33_n = np.nan
        verdict = "not a symmetry: the label of the original does not apply"

    fig, axes = plt.subplots(1, 2, figsize=(8.4, 4.4))
    for ax, im, t in [(axes[0], img, "original"), (axes[1], a, transform)]:
        ax.imshow(im, cmap="gray", interpolation="nearest")
        ax.set_title(t, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    plt.tight_layout(); plt.show()

    print(f"{'':16s} {'Vf':>8s} {'E22':>8s} {'E33':>8s} {'E_mean':>8s} {'dE':>8s}")
    print(f"{'original':16s} {img.mean():8.4f} {E22_o:8.3f} {E33_o:8.3f} "
          f"{(E22_o+E33_o)/2:8.3f} {E22_o-E33_o:+8.3f}")
    if np.isnan(E22_n):
        print(f"{'transformed':16s} {a.mean():8.4f} {'?':>8s} {'?':>8s} {'?':>8s} {'?':>8s}")
    else:
        print(f"{'transformed':16s} {a.mean():8.4f} {E22_n:8.3f} {E33_n:8.3f} "
              f"{(E22_n+E33_n)/2:8.3f} {E22_n-E33_n:+8.3f}")
    print(f"\nverdict: {verdict}")
    if show_S2:
        print(f"S2x - S2y:  original {sx0-sy0:+.4f}   transformed {sx1-sy1:+.4f}")

interact(transform_explorer,
         sample=IntSlider(int(i0), min=0, max=len(clean)-1, step=1, continuous_update=False),
         transform=Dropdown(options=list(TRANSFORMS.keys()), value="rotate 90 degrees"),
         show_S2=Checkbox(value=True, description="show S2"));


### Animation: the label stays put while the picture moves

The same cell rolled, mirrored and finally rotated. $E_{22}$, $E_{33}$ and $\Delta E$ are printed
underneath at every frame. Through the rolls and the mirrors they do not move, which is what makes
those frames free training data. At the rotation the two moduli exchange and $\Delta E$ flips
sign.

In [ ]:
# --- Animation: transformations with the target underneath -------------------
frames = []
for k in range(20):                                   # roll in x
    frames.append((("roll x", k * 3), "keep"))
for k in range(12):                                   # roll in y as well
    frames.append((("roll y", k * 5), "keep"))
for k in range(6):                                    # mirror in x
    frames.append((("mirror x", 0), "keep"))
for k in range(6):                                    # mirror in y as well
    frames.append((("mirror y", 0), "keep"))
for k in range(8):                                    # rotate 90
    frames.append((("rotate 90", 0), "swap"))

img_a = X_IMG[i0]
E22_a, E33_a = float(y_E22[i0]), float(y_E33[i0])

def build(f):
    a = img_a.copy(); shx = shy = 0; mx = my = False; rot = False
    for (op, amt), _ in frames[:f + 1]:
        if op == "roll x":   shx = amt
        if op == "roll y":   shy = amt
        if op == "mirror x": mx = True
        if op == "mirror y": my = True
        if op == "rotate 90": rot = True
    a = np.roll(np.roll(a, shx, axis=1), shy, axis=0)
    if mx: a = a[:, ::-1]
    if my: a = a[::-1, :]
    if rot: a = np.rot90(a, 1)
    return np.ascontiguousarray(a), rot

fig = plt.figure(figsize=(9.6, 4.6))
axI = fig.add_axes([0.03, 0.08, 0.42, 0.80])
axB = fig.add_axes([0.55, 0.18, 0.42, 0.62])
axI.set_xticks([]); axI.set_yticks([]); axI.grid(False)
imI = axI.imshow(img_a, cmap="gray", interpolation="nearest")
dE_a = E22_a - E33_a
bars = axB.bar(["$E_{22}$", "$E_{33}$", "$\\Delta E$"], [E22_a, E33_a, dE_a],
               color=[C_DATA, C_ALT, C_FIT], width=0.5)
axB.axhline(0, color="k", lw=0.9)
axB.set_ylim(min(-abs(dE_a) * 1.6, -1.0), max(E22_a, E33_a) * 1.25)
axB.set_ylabel("GPa")
txt = axB.set_title("", fontsize=10)

def update_aug(f):
    a, rot = build(f)
    imI.set_data(a)
    e22, e33 = (E33_a, E22_a) if rot else (E22_a, E33_a)
    bars[0].set_height(e22); bars[1].set_height(e33); bars[2].set_height(e22 - e33)
    op = frames[f][0][0]
    axI.set_title(f"transformation: {op}", fontsize=10)
    txt.set_text(f"$E_{{22}}$ = {e22:.3f},  $E_{{33}}$ = {e33:.3f} GPa\n"
                 f"$\\Delta E$ = {e22-e33:+.3f} GPa"
                 f"{'   (axes exchanged)' if rot else '   (unchanged)'}")
    return imI,

anim_aug = animation.FuncAnimation(fig, update_aug, frames=len(frames), interval=130, blit=False)
plt.close(fig)
print(f"{len(frames)} frames")
HTML(anim_aug.to_jshtml())


**What the animation shows.** Left: one cell being rolled, then mirrored, then rotated. Right:
the three quantities that would be used as its label, with the orange bar the anisotropy
$\Delta E = E_{22} - E_{33}$.

Through every roll and every mirror the three bars stand still while the picture moves. That is the
condition $y(T I) = y(I)$ holding, and it is what makes those frames free training data. At the
final frame the quarter turn exchanges the blue and green bars and the orange bar crosses zero. The
label did not become noisier, it became a different label.

### Measuring what each augmentation buys

Four runs on $\Delta E$, identical in every respect except the augmentation. Same architecture, same
split, same seed, same epoch count. The epoch count is short so that four runs fit in the budget, so
these scores sit below the longer runs in Notebook 3 and should be compared with each other rather
than with anything else.

The fourth run is the one to watch. It applies the 90 degree rotation and transforms the target with
it. The fifth, in the cell after, applies the same rotation and leaves the target alone, which is
the mistake.

In [ ]:
# --- Augmentation sweep on dE -------------------------------------------------
EPOCHS_AUG = 20 if DEVICE == "cpu" else 40      # short on purpose, see the text above
AUG_MODES  = ["none", "roll", "roll+flip", "roll+flip+rot"]

aug_res = {}
t_all = time.time()
for m in AUG_MODES:
    r = train_cnn(make_cnn(), X_IMG[itr], y_dE[itr], Xte_img, yd_te,
                  epochs=EPOCHS_AUG, aug=m, target_parity="odd", tag="dE, ")
    aug_res[m] = r
print(f"\nfour runs: {time.time() - t_all:.1f} s")


In [ ]:
# --- The same rotation, with the target left alone ---------------------------
r_wrong = train_cnn(make_cnn(), X_IMG[itr], y_dE[itr], Xte_img, yd_te,
                    epochs=EPOCHS_AUG, aug="roll+flip+rot_wrong", target_parity="odd",
                    tag="dE, WRONG rotation, ")
aug_res["roll+flip+rot_wrong"] = r_wrong


In [ ]:
# --- Augmentation results -----------------------------------------------------
labels = ["none", "roll", "roll+flip", "roll+flip+rot", "roll+flip+rot_wrong"]
pretty = ["no augmentation", "periodic roll", "roll + mirrors",
          "roll + mirrors + rot90\n(target swapped)", "roll + mirrors + rot90\n(target NOT swapped)"]
vals   = [aug_res[k]["r2"] for k in labels]
cols   = [C_ALT, C_DATA, C_DATA, C_DATA, C_BAD]

fig, ax = plt.subplots(figsize=(9.2, 4.4))
ax.bar(np.arange(len(vals)), vals, color=cols, width=0.6)
for i, v in enumerate(vals):
    ax.text(i, v + (0.02 if v >= 0 else -0.05), f"{v:+.3f}", ha="center", fontsize=9)
ax.set_xticks(np.arange(len(vals))); ax.set_xticklabels(pretty, fontsize=8)
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("test $R^2$ on $\\Delta E$")
ax.set_ylim(min(-0.15, min(vals) - 0.12), max(vals) + 0.15)
ax.set_title(f"the same CNN, {EPOCHS_AUG} epochs, only the augmentation differs", fontsize=10)
plt.tight_layout(); plt.show()

print(f"{'augmentation':36s} {'test R2':>9s} {'vs none':>9s} {'seconds':>9s}")
for k, p in zip(labels, [s.replace(chr(10), ' ') for s in pretty]):
    print(f"{p:36s} {aug_res[k]['r2']:>9.4f} {aug_res[k]['r2']-aug_res['none']['r2']:>+9.4f} "
          f"{aug_res[k]['elapsed']:>9.1f}")


### What the bars say

Read the first four bars against each other and the fifth against the fourth. The printed column
`vs none` is the measured gain of each augmentation over no augmentation at all, on this split, at
this epoch count, with one seed.

Two cautions before anyone quotes these numbers. Each bar is one seed, and Part 1 measured the
seed-to-seed spread of this architecture on this target at this training set size. Compare any gap
here against that spread before calling it an effect; on a 368 image test set most of these
differences will not survive the comparison. And augmentation trades bias for variance: it slows the
fit per epoch because each batch is a different set of pictures, so at a short budget a genuinely
useful augmentation can still look flat or worse.

The fifth bar is the one that does not need statistical care. Rotating the image and keeping the old
$\Delta E$ tells the network that a cell and its transpose have the same signed anisotropy. They do
not, they have opposite ones. Half the training signal is then being cancelled by the other half,
and the only consistent answer left is to predict near zero.

The general rule is worth stating plainly. An augmentation encodes a claimed invariance of the
target. If the claim is false, augmentation does not regularise the model, it corrupts the labels.
Derive the list from the symmetry of the problem, and check what each transformation does to the
quantity you are predicting, not just to the picture.

---

# Part 3 - Physical bounds on the output

A network trained on a regression target will happily return values that no two-phase composite can
have. When bounds are known, they can be imposed by construction rather than hoped for.

### What Voigt and Reuss actually bound

The Voigt and Reuss (Hill) results are statements about **tensors**, not about Young's modulus. For
a two-phase composite with phase stiffnesses $\mathbf{C}_m$, $\mathbf{C}_f$, compliances
$\mathbf{S}_m$, $\mathbf{S}_f$ and fibre volume fraction $v$, the minimum potential energy and
minimum complementary energy principles give

$$\boxed{\;\mathbf{S}_{\text{eff}} \;\le\; \mathbf{S}_{\text{R}} = (1-v)\mathbf{S}_m + v\mathbf{S}_f,
\qquad
\mathbf{C}_{\text{eff}} \;\le\; \mathbf{C}_{\text{V}} = (1-v)\mathbf{C}_m + v\mathbf{C}_f\;}$$

with $\le$ in the sense of quadratic forms. These bound the bulk and shear moduli directly, and they
are theorems.

Now specialise to Young's modulus. Apply the compliance bound to a uniaxial stress state: the
uniaxial compliance is $S_{1111} = 1/E$, so $1/E_{\text{eff}} \le (1-v)/E_m + v/E_f$, which
rearranges to

$$\boxed{\;E_{\text{eff}} \;\ge\; E_{\text{Reuss}}(v)
= \left(\frac{1-v}{E_m} + \frac{v}{E_f}\right)^{-1}\;}$$

That is rigorous, and it is the harmonic mean of the phase moduli weighted by volume.

The other side does not work the same way. The stiffness bound inverts to
$\mathbf{S}_{\text{eff}} \ge \mathbf{C}_{\text{V}}^{-1}$, which gives
$E_{\text{eff}} \le 1/(\mathbf{C}_{\text{V}}^{-1})_{1111}$. That quantity is not the arithmetic
rule of mixtures. The rule of mixtures

$$E_{\text{ROM}}(v) = (1-v)E_m + v E_f$$

is the **uniform strain estimate** of the axial modulus. It is not a rigorous upper bound on Young's
modulus, and the next cell demonstrates that with two exactly solvable composites that exceed it.

This matters here for a second reason. $E_{mean}$ is a **transverse** modulus, averaged over the two
in-plane directions perpendicular to the fibres. Uniform strain is the appropriate picture for the
**fibre** direction, where the phases carry load in parallel. Transversely the phases are much closer
to a series arrangement, so the response sits nearer the Reuss end. That is measured below.

The Hashin-Shtrikman bounds are tighter than Voigt and Reuss, because they use the additional
information that the composite is statistically isotropic in the transverse plane, but they are
written in terms of the bulk and shear moduli of the phases and therefore need the constituent
Poisson ratios. Those are not in this dataset, so Hashin-Shtrikman is mentioned and not used here.

### An honest complication

**The constituent properties are not in the dataset.** There is no column for $E_m$ or $E_f$. So
nothing here can be computed from first principles, and everything below is an inference from the
data.

What follows is two different things, and they must not be confused.

1. **Fitting the Reuss form and the rule of mixtures to the data.** Each has two free parameters, so
   each can be least-squares fitted to $E_{mean}$ against $v$. This produces two different pairs
   $(E_m, E_f)$, which is already the point: a single pair cannot make both curves pass through the
   data, because the data lie between the two.
2. **Choosing the pair that brackets the data.** The pair is chosen as the tightest one for which
   every training point lies inside the envelope. Call this what it is: an **empirical envelope**,
   fitted to the same data the model is trained on. It is not a theorem, it holds only where the
   data were, and Part 4 measures what happens when you leave that range.

In [ ]:
# --- Is the rule of mixtures an upper bound on E? Two exact counterexamples ---
# Both composites below can be solved in closed form, with no FE and no approximation.
#
# (a) A layered composite, layers normal to x3, loaded uniaxially along x1, in the plane
#     of the layers. Compatibility makes e11 and e22 uniform across layers, equilibrium
#     makes s33 = 0 in every layer, so e33 follows layer by layer and the 2x2 system for
#     (e11, e22) is exact.
# (b) Hashin's composite cylinder assemblage axial modulus, the exact result for a
#     unidirectional composite loaded along the fibres.

def E_layered_inplane(Em, nm, Ef, nf, v):
    def lam_G(E, nu):
        return E*nu/((1+nu)*(1-2*nu)), E/(2*(1+nu))
    A = np.zeros((2, 2))
    for (E_, nu_), fr in [((Em, nm), 1-v), ((Ef, nf), v)]:
        lam, G = lam_G(E_, nu_)
        a = lam/(lam + 2*G)                     # e33 = -a (e11 + e22), from s33 = 0
        A[0, 0] += fr*(lam*(1-a) + 2*G); A[0, 1] += fr*(lam*(1-a))
        A[1, 0] += fr*(lam*(1-a));       A[1, 1] += fr*(lam*(1-a) + 2*G)
    return 1.0/np.linalg.solve(A, np.array([1.0, 0.0]))[0]

def E_axial_cca(Em, nm, Ef, nf, v):
    k = lambda E, nu: E/(2*(1+nu)*(1-2*nu))     # plane-strain bulk modulus
    Gm = Em/(2*(1+nm))
    return (v*Ef + (1-v)*Em
            + 4*v*(1-v)*(nf-nm)**2/(v/k(Em, nm) + (1-v)/k(Ef, nf) + 1.0/Gm))

print("exact E of a two-phase composite against the rule of mixtures")
print(f"{'Em':>5s} {'nu_m':>5s} {'Ef':>6s} {'nu_f':>5s} {'v':>4s} "
      f"{'Reuss':>8s} {'ROM':>9s} {'exact layered':>14s} {'exact CCA axial':>16s}")
for Em_, nm_, Ef_, nf_, v_ in [(3.5, 0.35, 20.0, 0.20, 0.5),
                               (3.5, 0.40, 70.0, 0.22, 0.5),
                               (1.0, 0.45, 10.0, 0.10, 0.5),
                               (3.5, 0.30, 20.0, 0.30, 0.5)]:
    rom = (1-v_)*Em_ + v_*Ef_
    rss = 1.0/((1-v_)/Em_ + v_/Ef_)
    print(f"{Em_:>5.1f} {nm_:>5.2f} {Ef_:>6.1f} {nf_:>5.2f} {v_:>4.1f} "
          f"{rss:>8.4f} {rom:>9.4f} {E_layered_inplane(Em_,nm_,Ef_,nf_,v_):>14.4f} "
          f"{E_axial_cca(Em_,nm_,Ef_,nf_,v_):>16.4f}")
print()
print("Both exact columns exceed the rule of mixtures on every row where the constituent")
print("Poisson ratios differ, and equal it exactly on the last row where they do not.")
print("The Hashin axial result is ROM + 4 v (1-v) (nu_f - nu_m)^2 / (...), and that extra")
print("term is non-negative, so the rule of mixtures is an estimate, not an upper bound.")
print("The Reuss column is below the exact answer everywhere, as the theorem requires.")


**What the table shows.** Four two-phase composites, each solved exactly. The last row has
equal Poisson ratios in the two phases and the exact answers land on the rule of mixtures. The first
three have mismatched Poisson ratios and both exact answers sit above it.

The mechanism is in Hashin's axial result: the exact modulus is the rule of mixtures plus a term
proportional to $(\nu_f - \nu_m)^2$, which is non-negative. Under axial load the phases want to
contract laterally by different amounts, they constrain each other, and the constraint stiffens the
composite beyond the parallel-spring estimate.

None of this rescues the rule of mixtures as a bound, and it is worth being clear about the size of
the effect: the excess in the table is a fraction of a percent. The rule of mixtures is an excellent
estimate for the fibre direction. It is simply not a theorem, and it is not the right picture for
the transverse direction at all.

In [ ]:
# --- Fitting the rule of mixtures and the Reuss form to the data -------------
from scipy.optimize import curve_fit

def voigt(v, Em, Ef): return (1 - v) * Em + v * Ef      # rule of mixtures, uniform strain
def reuss(v, Em, Ef): return 1.0 / ((1 - v) / Em + v / Ef)

v_tr, E_tr = VF[itr].astype(np.float64), y_mean[itr].astype(np.float64)

p_v, _ = curve_fit(voigt, v_tr, E_tr, p0=[4.0, 20.0], maxfev=20000)
rms_v  = np.sqrt(((voigt(v_tr, *p_v) - E_tr) ** 2).mean())

print("least squares fit of each form to the training data")
print(f"  rule of mixtures : Em = {p_v[0]:7.3f} GPa   Ef = {p_v[1]:8.3f} GPa   "
      f"rms {rms_v:.3f} GPa")

# The Reuss form is NOT identified on this data: the fit is almost flat in Ef and runs
# away to whatever upper limit the optimiser is given. Show that rather than hide it.
print("  Reuss form       : Ef is not identified by this data. Raising the optimiser's")
print("                     upper limit moves the fitted Ef with it:")
print(f"{'':21s} {'limit on Ef':>13s} {'fitted Em':>11s} {'fitted Ef':>12s} {'rms GPa':>9s}")
for ub in [1e2, 1e4, 1e6, 1e8]:
    pr, _ = curve_fit(reuss, v_tr, E_tr, p0=[4.0, 60.0], maxfev=50000,
                      bounds=([0.1, 1.0], [20.0, ub]))
    print(f"{'':21s} {ub:>13.0e} {pr[0]:>11.3f} {pr[1]:>12.3e} "
          f"{np.sqrt(((reuss(v_tr,*pr)-E_tr)**2).mean()):>9.3f}")
p_r = pr
print()
print("Two things follow. The transverse data are far above a Reuss curve through the")
print("matrix modulus, so the fit can only approach them by sending Ef to infinity, and")
print("the fitted Ef is then an artefact of the limit rather than a material property.")
print("And a curve least-squares fitted through the middle of the data is not a bound")
print("on that data, whichever form it has.")


In [ ]:
# --- The tightest envelope that actually contains the training data ----------
def tightest_envelope(v, E, n_grid=400):
    # smallest mean-width Reuss / rule-of-mixtures envelope containing every training point
    best = None
    for Em in np.linspace(1.0, float(E.min()) * 0.999, n_grid):
        ef_lo = np.max((E - (1 - v) * Em) / v)            # ROM curve must sit above the data
        den   = 1.0 / E - (1 - v) / Em
        pos   = den > 0
        ef_hi = np.min(v[pos] / den[pos]) if pos.any() else np.inf   # Reuss must sit below
        if ef_lo > ef_hi:
            continue
        for Ef in (ef_lo, ef_hi):
            if not np.isfinite(Ef):
                continue
            w = float(np.mean(voigt(v, Em, Ef) - reuss(v, Em, Ef)))
            if best is None or w < best[0]:
                best = (w, Em, Ef)
    return best

W_BAND, EM_HAT, EF_HAT = tightest_envelope(v_tr, E_tr)
lo_tr, hi_tr = reuss(v_tr, EM_HAT, EF_HAT), voigt(v_tr, EM_HAT, EF_HAT)

print(f"fitted constituents (NOT given in the dataset):")
print(f"  matrix  Em = {EM_HAT:.3f} GPa")
print(f"  fibre   Ef = {EF_HAT:.3f} GPa")
print(f"training points inside the envelope: {np.mean((E_tr >= lo_tr-1e-9) & (E_tr <= hi_tr+1e-9)):.3f}")
print(f"band width: mean {W_BAND:.2f} GPa, from {np.min(hi_tr-lo_tr):.2f} to "
      f"{np.max(hi_tr-lo_tr):.2f} GPa")
print(f"spread of E_mean itself: std {y_mean.std():.2f} GPa, range "
      f"{y_mean.min():.2f} to {y_mean.max():.2f} GPa")

# Where inside the envelope does the transverse data actually sit?
# 0 is on the Reuss curve, 1 is on the rule of mixtures.
loA, hiA = reuss(VF.astype(np.float64), EM_HAT, EF_HAT), voigt(VF.astype(np.float64), EM_HAT, EF_HAT)
posn = (y_mean - loA) / (hiA - loA)
print(f"\nposition inside the envelope, 0 = Reuss curve, 1 = rule of mixtures")
print(f"{'vol frac':>9s} {'Reuss':>8s} {'data mean':>10s} {'ROM':>8s} {'position':>9s}")
for vfq in sorted(clean.vol_frac.unique()):
    m = clean.vol_frac.values == vfq
    print(f"{vfq:>8.0f}% {reuss(vfq/100, EM_HAT, EF_HAT):>8.2f} {y_mean[m].mean():>10.2f} "
          f"{voigt(vfq/100, EM_HAT, EF_HAT):>8.2f} {posn[m].mean():>9.3f}")
print(f"\nall microstructures: mean position {posn.mean():.3f}")


In [ ]:
# --- The envelope against the data -------------------------------------------
vv = np.linspace(0.05, 0.65, 200)
fig, ax = plt.subplots(figsize=(7.4, 4.6))
ax.fill_between(vv, reuss(vv, EM_HAT, EF_HAT), voigt(vv, EM_HAT, EF_HAT),
                color=C_ALT, alpha=0.18, label="empirical envelope, fitted constituents")
ax.plot(vv, voigt(vv, EM_HAT, EF_HAT), color=C_ALT, lw=1.8, label="rule of mixtures (upper)")
ax.plot(vv, reuss(vv, EM_HAT, EF_HAT), color=C_ALT, lw=1.8, ls="--", label="Reuss (lower)")
ax.plot(vv, voigt(vv, *p_v), color=C_BAD, lw=1.4, ls=":",
        label="rule of mixtures least-squares fitted to the data")
ax.scatter(VF + np.random.RandomState(0).normal(0, 0.004, len(VF)), y_mean,
           s=8, alpha=0.35, color=C_DATA, label="microstructures")
ax.set_xlabel("fibre volume fraction $v$"); ax.set_ylabel("$E_{mean}$ (GPa)")
ax.set_title("empirical envelope from fitted constituent moduli", fontsize=10)
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()


**What the figure shows.** Blue points are the microstructures kept after dropping outliers.
The shaded band between the solid and dashed orange curves is the empirical envelope from the fitted
constituents, and the dotted red curve is the rule of mixtures least-squares fitted through the data,
which is why it cuts straight through the middle of the cloud instead of sitting above it. A curve
fitted to minimise error is not a bound on the data it was fitted to.

The orange band is wide. Its mean width is printed above, and it is several GPa across most of
the range, while the data at a fixed volume fraction spread over a fraction of that. These are weak
constraints, which is the usual situation and the reason Hashin-Shtrikman exists.

### Where the data sit in the band, and why

The position table printed above is the physical reading. At 10 percent fibre the transverse data sit
almost on the Reuss curve; the position climbs steadily with volume fraction. That is what transverse
loading of a unidirectional composite looks like. Load applied perpendicular to the fibres passes
through matrix and fibre much more nearly in series than in parallel, so the compliant phase
dominates and the uniform strain picture is the wrong one.

Two warnings about reading too much into the exact numbers. The envelope was squeezed onto this data,
so the position at the highest volume fraction is pushed towards 1 by construction. And the position
is a property of the fitted constituents, not of the material. The direction of the trend is the
robust part.

### Building the bound into the network

The volume fraction is not a separate input here, it is the mean of the image, so the network can
compute the bracket for itself. Let

$$\hat v (I) \;=\; \frac{1}{n^2}\sum_{i,j} I(i,j), \qquad n = 64$$

be the white pixel fraction of the image, which is the volume fraction. Write $z(I)$ for the
unconstrained scalar the convolutional body produces, and replace the final linear output by

$$\boxed{\;\hat E(I) \;=\; E_{\text{Reuss}}(\hat v) \;+\;
\sigma\bigl(z(I)\bigr)\,\big[E_{\text{Voigt}}(\hat v) - E_{\text{Reuss}}(\hat v)\big], \qquad
\sigma(z) = \frac{1}{1 + e^{-z}}\;}$$

$\sigma$ is the logistic function from Notebook 1, mapping the whole real line into $(0,1)$, so
$\hat E$ is a convex combination of the two curves. The prediction is then inside the envelope for
every input, by construction, for any value of the weights, and the output is in GPa rather than in
standardised units.

### Designing the comparison so it measures the right thing

The bounded head does three things at once, and only one of them is the constraint.

1. It **constrains** the output to the envelope.
2. It supplies an **output offset**. A plain network trained on the raw target in GPa starts with its
   final layer near zero and has to travel to a mean of about 7 GPa. The bounded head starts at
   $\sigma(0)$, the midpoint of the envelope, which is already the right magnitude.
3. It supplies a **volume fraction prior**. The envelope moves with $\hat v$, which the head reads
   off the image, so the first-order dependence on volume fraction is built in before any training.

A two-way comparison of bounded against a plain network on the raw GPa target would credit the
constraint with all three. So three models are trained below, identical in architecture, data,
augmentation, seed and epochs:

- **unbounded, raw GPa target.** The naive baseline, and the one that isolates effects 2 and 3.
- **unbounded, standardised target.** The target is centred and scaled as everywhere else in these
  notebooks, which removes effect 2. This is the fair baseline.
- **bounded output.** All three effects.

The difference between the first two is the output offset. The difference between the second and the
third is what the constraint is worth. That is the number the section is actually about.

In [ ]:
# --- A bounded output head ----------------------------------------------------
class Bounded(nn.Module):
    # squashes the network output into the Voigt-Reuss band for the image own Vf
    def __init__(self, body, Em, Ef):
        super().__init__()
        self.body = body
        self.Em, self.Ef = float(Em), float(Ef)

    def forward(self, x):
        v  = x.mean(dim=(1, 2, 3), keepdim=False).clamp(1e-3, 1 - 1e-3).view(-1, 1)
        hi = (1 - v) * self.Em + v * self.Ef
        lo = 1.0 / ((1 - v) / self.Em + v / self.Ef)
        return lo + torch.sigmoid(self.body(x)) * (hi - lo)

EPOCHS_BND = 20 if DEVICE == "cpu" else 40
ym_tr = y_mean[itr]

res_raw = train_cnn(make_cnn(), X_IMG[itr], ym_tr, Xte_img, ym_te,
                    epochs=EPOCHS_BND, aug="roll+flip", target_parity="even",
                    standardise=False, tag="E_mean, unbounded raw GPa target,   ")

res_free = train_cnn(make_cnn(), X_IMG[itr], ym_tr, Xte_img, ym_te,
                     epochs=EPOCHS_BND, aug="roll+flip", target_parity="even",
                     standardise=True, tag="E_mean, unbounded standardised,     ")


In [ ]:
# --- The same network with the bound imposed ---------------------------------
res_bnd = train_cnn(Bounded(make_cnn(), EM_HAT, EF_HAT), X_IMG[itr], ym_tr, Xte_img, ym_te,
                    epochs=EPOCHS_BND, aug="roll+flip", target_parity="even",
                    standardise=False, tag="E_mean, bounded output,             ")

# how often does each model leave the envelope?
vf_te  = X_IMG[ite].mean(axis=(1, 2)).astype(np.float64)
lo_te  = reuss(vf_te, EM_HAT, EF_HAT); hi_te = voigt(vf_te, EM_HAT, EF_HAT)

print()
print(f"{'model':30s} {'test R2':>9s} {'RMSE GPa':>10s} {'outside envelope':>18s}")
ROWS = [("unbounded, raw GPa target", res_raw),
        ("unbounded, standardised", res_free),
        ("bounded output", res_bnd)]
for nm, r in ROWS:
    ob = float(np.mean((r["pred"] < lo_te) | (r["pred"] > hi_te)))
    rmse = float(np.sqrt(((r["pred"] - ym_te) ** 2).mean()))
    print(f"{nm:30s} {r['r2']:>9.4f} {rmse:>10.3f} {ob:>17.1%}")
print()
print(f"R2 from standardising the target alone, no bound : "
      f"{res_free['r2'] - res_raw['r2']:+.4f}")
print(f"R2 from adding the bound on top of that          : "
      f"{res_bnd['r2'] - res_free['r2']:+.4f}")
print(f"R2 of bounded against the naive raw-target model  : "
      f"{res_bnd['r2'] - res_raw['r2']:+.4f}")


In [ ]:
# --- Predictions against the envelope -----------------------------------------
fig, axes3 = plt.subplots(1, 3, figsize=(14.5, 4.3), sharey=True)
o = np.argsort(vf_te)
for ax, r, t in [(axes3[0], res_raw,  "unbounded, raw GPa target"),
                 (axes3[1], res_free, "unbounded, standardised"),
                 (axes3[2], res_bnd,  "bounded output")]:
    ax.fill_between(vf_te[o], lo_te[o], hi_te[o], color=C_ALT, alpha=0.18)
    ax.scatter(vf_te, ym_te, s=10, alpha=0.35, color="0.5", label="true")
    ax.scatter(vf_te, r["pred"], s=10, alpha=0.6, color=C_DATA, label="predicted")
    ax.set_xlabel("fibre volume fraction from the image")
    ax.set_title(f"{t}\ntest $R^2$ = {r['r2']:.4f}", fontsize=9.5)
axes3[0].set_ylabel("$E_{mean}$ (GPa)")
axes3[0].legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()


**What the three panels show.** Each plots the test set against the volume fraction read off
the image, with the shaded region the empirical envelope. Grey is the true $E_{mean}$, blue is the
prediction. Left is the naive baseline on the raw GPa target, middle is the same network on a
standardised target, right has the bound built into the output layer. The test $R^2$ of each is in
its title and the printed table gives the fraction of predictions that fell outside the envelope.

Read the left panel against the middle one first. Most of the apparent benefit of the bounded head
is not the bound, it is the output offset: standardising the target, with no constraint anywhere,
recovers most of the gap and takes the fraction of out-of-envelope predictions down on its own. That
is worth knowing on its own account, because it is a one-line change that costs nothing.

The middle panel against the right one is what the bound is worth. The printed difference is small,
and the reason is in the envelope width printed earlier: the network's error is already far below it,
so the constraint is slack almost everywhere and has little left to fix.

### Interactive: how much the constituents matter

The fitted $E_m$ and $E_f$ set where the band sits. Move them and watch what fraction of the data
falls inside. A band that contains everything with room to spare constrains nothing, and a band that
excludes real data is worse than useless because it will clip correct predictions.

In [ ]:
# --- Interactive bound explorer ----------------------------------------------
def bound_explorer(Em=float(round(EM_HAT, 2)), Ef=float(round(EF_HAT, 1))):
    lo, hi = reuss(VF, Em, Ef), voigt(VF, Em, Ef)
    inside = float(np.mean((y_mean >= lo) & (y_mean <= hi)))
    width  = float(np.mean(hi - lo))

    fig, ax = plt.subplots(figsize=(7.0, 4.2))
    ax.fill_between(vv, reuss(vv, Em, Ef), voigt(vv, Em, Ef), color=C_ALT, alpha=0.18)
    ax.plot(vv, voigt(vv, Em, Ef), color=C_ALT, lw=1.6)
    ax.plot(vv, reuss(vv, Em, Ef), color=C_ALT, lw=1.6, ls="--")
    ok = (y_mean >= lo) & (y_mean <= hi)
    ax.scatter(VF[ok],  y_mean[ok],  s=8, alpha=0.35, color=C_DATA)
    ax.scatter(VF[~ok], y_mean[~ok], s=14, alpha=0.8, color=C_BAD)
    ax.set_xlim(0.03, 0.67); ax.set_ylim(0, max(16, voigt(0.65, Em, Ef) * 1.05))
    ax.set_xlabel("fibre volume fraction"); ax.set_ylabel("$E_{mean}$ (GPa)")
    ax.set_title(f"inside the band: {inside:.1%}   mean width {width:.2f} GPa", fontsize=10)
    plt.tight_layout(); plt.show()
    print(f"red points violate the band and would be clipped by a bounded model: "
          f"{int((~ok).sum())} of {len(ok)}")

interact(bound_explorer,
         Em=FloatSlider(float(round(EM_HAT, 2)), min=1.0, max=6.0, step=0.05,
                        continuous_update=False),
         Ef=FloatSlider(float(round(EF_HAT, 1)), min=8.0, max=60.0, step=0.5,
                        continuous_update=False));


### The honest verdict on this experiment

The printed comparison above is the answer, whatever it happens to be on your run. Read the three
differences printed under the table in order, and quote the middle one as the effect of the bound.

Two things are worth taking from it. A naive regression head on an unscaled physical target wastes a
substantial part of a short training budget learning the mean, and that shows up as a real accuracy
loss and as predictions outside a physically admissible range. And once that is fixed, the
constraint has very little left to do on this dataset, because the model's error is already far
smaller than the width of the envelope.

Resist the temptation to describe the remaining change as an improvement unless it is larger than
the run-to-run spread measured in Part 1.

Where this technique does earn its place is different from raw accuracy.

- A surrogate inside an optimiser gets pushed into corners of the input space where it was never
  trained. A hard bound is what stops the optimiser exploiting a region of nonsense.
- The bound is worth most when the model is weak, the data are few, or the target is being
  extrapolated. Part 4 is about exactly that case.
- The bound is only as good as the constituents used to build it. Here they were fitted to the
  training data, so the envelope inherits the training range. That limitation is measured in Part 4.
- A rigorous bound would behave differently. The Reuss lower curve is a theorem once the constituents
  are known; the upper curve here is an estimate fitted to data. Imposing a fitted envelope as a hard
  constraint is only safe where the fit was made.

---

# Part 4 - Where it fails: outside the training range

Every score so far comes from a random split, so the test microstructures were drawn from the same
design of experiment as the training ones: volume fractions 10 to 60 percent, diameters 8, 10 and 12.
A random split measures interpolation.

Surrogate models are rarely used that way. They get built on the runs somebody had time to do, and
then asked about a design nobody ran. So the test that matters is a split by a design variable,
not a random one.

The experiment: train on volume fraction 10 to 40 percent only, then predict at 50 and 60 percent.
Three models get the same treatment, the CNN on the raw image, the gradient boosting model on the 14
the 14 named descriptors from Notebook 2, and a linear model in $v$ and $v^2$ from Notebook 1.

In [ ]:
# --- The extrapolation split --------------------------------------------------
in_rng  = np.where(clean.vol_frac.values <= 40)[0]
out_rng = np.where(clean.vol_frac.values >= 50)[0]
itr_ood, ite_ood = train_test_split(in_rng, test_size=0.25, random_state=SEED)

print(f"train      vf 10-40 percent : {len(itr_ood)} microstructures")
print(f"test  (in) vf 10-40 percent : {len(ite_ood)}")
print(f"test (out) vf 50-60 percent : {len(out_rng)}")
print(f"\nE_mean, training range  {y_mean[itr_ood].min():.2f} to {y_mean[itr_ood].max():.2f} GPa")
print(f"E_mean, extrapolation   {y_mean[out_rng].min():.2f} to {y_mean[out_rng].max():.2f} GPa")


In [ ]:
# --- CNN trained inside the range only ---------------------------------------
EPOCHS_OOD = 20 if DEVICE == "cpu" else 40

res_ood = train_cnn(make_cnn(), X_IMG[itr_ood], y_mean[itr_ood],
                    X_IMG[ite_ood], y_mean[ite_ood],
                    epochs=EPOCHS_OOD, aug="roll+flip", target_parity="even",
                    tag="E_mean, vf<=40 only, ")

mu_o, sd_o = float(y_mean[itr_ood].mean()), float(y_mean[itr_ood].std())
pred_cnn_out = predict(res_ood["model"], X_IMG[out_rng], mu_o, sd_o)


In [ ]:
# --- Descriptors and a physics-shaped linear model, same split ---------------
t0 = time.time()
gb = HistGradientBoostingRegressor(random_state=SEED).fit(X_DESC[itr_ood], y_mean[itr_ood])
gb_in  = gb.predict(X_DESC[ite_ood]); gb_out = gb.predict(X_DESC[out_rng])
print(f"descriptor gradient boosting fitted in {time.time()-t0:.1f} s")

def poly_feats(v): return np.c_[v, v**2]
lin = LinearRegression().fit(poly_feats(VF[itr_ood]), y_mean[itr_ood])
lin_in  = lin.predict(poly_feats(VF[ite_ood])); lin_out = lin.predict(poly_feats(VF[out_rng]))

def report(name, p_in, p_out):
    r2_in  = r2_score(y_mean[ite_ood], p_in)
    r2_out = r2_score(y_mean[out_rng], p_out)
    rm_in  = float(np.sqrt(((p_in  - y_mean[ite_ood]) ** 2).mean()))
    rm_out = float(np.sqrt(((p_out - y_mean[out_rng]) ** 2).mean()))
    bias   = float((p_out - y_mean[out_rng]).mean())
    print(f"{name:26s} {r2_in:>8.4f} {rm_in:>9.3f} {r2_out:>9.4f} {rm_out:>9.3f} {bias:>+9.3f}")
    return dict(r2_in=r2_in, r2_out=r2_out, rmse_in=rm_in, rmse_out=rm_out, bias=bias,
                p_in=p_in, p_out=p_out)

print(f"\n{'model':26s} {'R2 in':>8s} {'RMSE in':>9s} {'R2 out':>9s} {'RMSE out':>9s} "
      f"{'bias out':>9s}")
OOD = {}
OOD["CNN on the image"]      = report("CNN on the image", res_ood["pred"], pred_cnn_out)
OOD["14 descriptors, GB"]    = report("14 descriptors, GB", gb_in, gb_out)
OOD["linear in v and v^2"]   = report("linear in v and v^2", lin_in, lin_out)
print("\nbias out is the mean signed error at vf 50-60. Negative means under-prediction.")


In [ ]:
# --- The picture of the failure ----------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2), sharey=True)
vf_in, vf_out = VF[ite_ood], VF[out_rng]
for ax, (nm, d) in zip(axes, OOD.items()):
    ax.axvspan(0.05, 0.45, color=C_ALT, alpha=0.10)
    ax.scatter(vf_in,  y_mean[ite_ood], s=10, color="0.55", alpha=0.5, label="true")
    ax.scatter(vf_out, y_mean[out_rng], s=10, color="0.55", alpha=0.5)
    ax.scatter(vf_in,  d["p_in"],  s=10, color=C_DATA, alpha=0.6, label="predicted, in range")
    ax.scatter(vf_out, d["p_out"], s=12, color=C_BAD,  alpha=0.7, label="predicted, extrapolated")
    ax.set_xlabel("fibre volume fraction")
    ax.set_title(f"{nm}\n$R^2$ in {d['r2_in']:.3f}, out {d['r2_out']:+.3f}", fontsize=9)
axes[0].set_ylabel("$E_{mean}$ (GPa)")
axes[0].legend(fontsize=7, loc="upper left")
axes[0].text(0.25, 13.0, "trained here", fontsize=8, ha="center", color=C_ALT)
plt.tight_layout(); plt.show()


**What the three panels show.** One panel per model, the same axes in each. The shaded strip
on the left of every panel is the volume fraction range the model was trained on. Grey is the truth,
blue is the prediction inside that range, red is the prediction at 50 and 60 percent where no
training data existed. The two $R^2$ values in each title are the in-range and the extrapolated
score.

Look at the red points against the grey ones they are supposed to sit on. The failure is not extra
scatter, it is a shift: the red clouds sit below the truth, by an amount that grows with distance
from the training range. The bias column printed above is that shift as a single number per model.

In [ ]:
# --- How badly, in one number per model --------------------------------------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4))
names = list(OOD.keys()); x = np.arange(len(names)); w = 0.36
a1.bar(x - w/2, [OOD[n]["rmse_in"]  for n in names], w, color=C_DATA, label="inside 10-40%")
a1.bar(x + w/2, [OOD[n]["rmse_out"] for n in names], w, color=C_BAD,  label="at 50-60%")
a1.set_xticks(x); a1.set_xticklabels([n.replace(" ", "\n", 1) for n in names], fontsize=8)
a1.set_ylabel("RMSE on $E_{mean}$ (GPa)"); a1.legend(fontsize=8)
a1.set_title("error inside and outside the training range", fontsize=10)

ratio = [OOD[n]["rmse_out"] / OOD[n]["rmse_in"] for n in names]
a2.bar(x, ratio, 0.5, color=[C_BAD if r > 3 else C_ALT for r in ratio])
for i, r in enumerate(ratio):
    a2.text(i, r * 1.03, f"{r:.0f}x", ha="center", fontsize=9)
a2.set_xticks(x); a2.set_xticklabels([n.replace(" ", "\n", 1) for n in names], fontsize=8)
a2.set_ylabel("RMSE ratio, outside / inside")
a2.set_title("how much worse extrapolation is", fontsize=10)
plt.tight_layout(); plt.show()


**What the two bar charts show.** Left: the RMSE of each model inside the training range in
blue and at 50 to 60 percent in red, in GPa, so the two are directly comparable. Right: the ratio of
those two numbers, printed on each bar.

Read the ratio and then read it again. Every one of these models scored well on a random split, and
every one of them is reporting a test score that says nothing whatever about the regime it has just
been asked about.

### What went wrong, and why it was predictable

The gradient boosting model is the clearest case. A tree splits on thresholds, and outside the range
of the training data there are no further splits, so every prediction saturates at the value of the
last leaf. The model is not wrong about the data it saw, it simply has no mechanism for going
higher. The bias column shows this as a large one-sided under-prediction.

The CNN does not saturate quite so hard, because the final layer is affine in the learned features,
but it is still fitting a flexible function to a range it never visited. Both flexible models have a
large error ratio between the extrapolated and the in-range test sets.

The linear model in $v$ and $v^2$ does noticeably better outside the range, and it is the least
accurate of the three inside it. That is the trade in plain form. A model with a functional shape
taken from the physics has far fewer ways to be wrong when it leaves the data, and a model with
enough capacity to fit anything inside the range has correspondingly many.

### The fitted bound does not rescue this

Part 3 built an empirical envelope from constituents fitted to the training data. If the training data
stops at 40 percent, so does the evidence for those constituents. The cell below refits the envelope
on the extrapolation training set alone, then checks it at 50 and 60 percent.

In [ ]:
# --- Refit the envelope inside the range, then test it outside ---------------
Em_o, Ef_o = tightest_envelope(VF[itr_ood].astype(np.float64),
                               y_mean[itr_ood].astype(np.float64))[1:]
print(f"constituents fitted on vf 10-40 only: Em = {Em_o:.3f} GPa, Ef = {Ef_o:.3f} GPa")
print(f"constituents fitted on the full range: Em = {EM_HAT:.3f} GPa, Ef = {EF_HAT:.3f} GPa\n")

print(f"{'vol frac':>9s} {'Reuss':>8s} {'ROM':>8s} {'data min':>9s} {'data max':>9s} "
      f"{'inside':>8s}")
for vfq in [10, 20, 30, 40, 50, 60]:
    sel = clean.vol_frac.values == vfq
    vq  = vfq / 100.0
    lo, hi = reuss(vq, Em_o, Ef_o), voigt(vq, Em_o, Ef_o)
    ins = float(np.mean((y_mean[sel] >= lo) & (y_mean[sel] <= hi)))
    print(f"{vfq:>9d} {lo:>8.2f} {hi:>8.2f} {y_mean[sel].min():>9.2f} "
          f"{y_mean[sel].max():>9.2f} {ins:>7.1%}")


The last two rows are the lesson. A bound inferred from data is only a bound where the data
were. Taken outside the fitted range the envelope can exclude perfectly real microstructures, and a
bounded network built on it would then clip correct predictions with complete confidence.

If the constituent properties are genuinely known, from the material datasheet rather than from a
fit, the bound does hold everywhere and this objection disappears. Knowing which of those two
situations you are in is the whole point of this section.

### What to do about extrapolation

There is no trick that makes a fitted model safe outside its data. The practical measures are
mundane and they work.

1. Record the training range of every input variable and refuse to predict outside it, or flag the
   prediction loudly when the user insists.
2. Split by a design variable, not at random, when reporting how a surrogate will behave in use. A
   random split flatters every model in this section.
3. Prefer a model whose functional form comes from the physics when extrapolation is unavoidable,
   and accept the accuracy it costs inside the range.
4. Use real bounds when the constituents are known. Use fitted bounds only inside the fitted range.

### The sentence to remember from this part

A surrogate has no way of telling you that it is extrapolating. It returns a number of the usual
size, in the usual units, with the usual number of decimal places, and the printed comparison above
is what that number was worth. Nothing in the model's own output distinguished the red points from
the blue ones. The only defence is knowing the training range and refusing to go outside it, and
that is a bookkeeping job for the person who built the model, not something the model can do for
itself.

Part 5 makes this sharper still. The ensemble there is the obvious candidate for a warning system,
and it would not have raised one here.

---

# Part 5 - Uncertainty, cheaply

A single number from a surrogate is not usable in a design decision. Something has to say how much
to trust it.

The full Bayesian treatments are expensive. A **deep ensemble** is the cheap version that works: train
the same architecture several times from different random initialisations, then use the mean of the
members as the prediction and their standard deviation as an uncertainty estimate. Nothing else
changes, and the cost is the number of members.

It works because non-convex optimisation from different starting points lands in different minima.
Where the data pin the function down, the members agree. Where the data are sparse or the target is
noisy, they diverge. That divergence is the signal.

### The two quantities, defined

Train $M$ copies of the same architecture on the same data, differing only in the random seed $s$,
and write $\hat y^{(s)}(I)$ for the prediction of member $s$ on input $I$. The prediction reported is
the ensemble mean and the uncertainty reported is the ensemble standard deviation:

$$\boxed{\;\bar{y}(I) = \frac{1}{M}\sum_{s=1}^{M}\hat y^{(s)}(I), \qquad
u(I) = \left[\frac{1}{M}\sum_{s=1}^{M}\bigl(\hat y^{(s)}(I) - \bar{y}(I)\bigr)^2\right]^{1/2}\;}$$

Both have the units of the target, GPa. $M = 4$ below. $u$ is a spread between members, not a
posterior standard deviation, and the difference matters: it is worth what the tests in this part
say it is worth and no more.

Four members are trained below on $\Delta E$ with a short epoch budget, which keeps the whole
section inside the time budget. Four is the minimum that gives a usable spread. Five to ten is more
typical in practice.

In [ ]:
# --- A small deep ensemble on dE ---------------------------------------------
N_MEMBERS  = 4
EPOCHS_ENS = 16 if DEVICE == "cpu" else 32

ens_pred, ens_r2 = [], []
t_all = time.time()
for s in range(N_MEMBERS):
    r = train_cnn(make_cnn(seed=100 + s), X_IMG[itr], y_dE[itr], Xte_img, yd_te,
                  epochs=EPOCHS_ENS, aug="roll+flip", target_parity="odd",
                  seed=100 + s, tag=f"member {s}, ")
    ens_pred.append(r["pred"]); ens_r2.append(r["r2"])
ens_pred = np.array(ens_pred)
print(f"\nensemble of {N_MEMBERS}: {time.time() - t_all:.1f} s")

mean_pred = ens_pred.mean(axis=0)
spread    = ens_pred.std(axis=0)
R2_ENS    = r2_score(yd_te, mean_pred)

print(f"\nindividual members: " + ", ".join(f"{r:+.4f}" for r in ens_r2))
print(f"mean of members:    {np.mean(ens_r2):+.4f}")
print(f"ensemble mean:      {R2_ENS:+.4f}   "
      f"({R2_ENS - np.mean(ens_r2):+.4f} over the average member)")
print(f"spread: median {np.median(spread):.3f} GPa, "
      f"range {spread.min():.3f} to {spread.max():.3f} GPa")


Averaging the members is worth reporting on its own. The ensemble mean is usually better than
the average individual member, because independent errors partly cancel. The printed difference is
the size of that effect here.

Look at the individual member scores before reading that difference. Part 1 measured how far runs of
this architecture move with the seed alone, and the same thing happens here: if one member has landed
near $R^2 = 0$ it has learned little beyond the training mean, and it drags the average member score
down while inflating the spread at every test point. That is worth knowing, because it means the
uncertainty estimate below is partly reporting optimiser failure rather than genuine ambiguity in the
data. Both are real sources of doubt about a prediction, but they are not the same thing.

### Does the spread track the error?

This is the claim that has to be tested rather than assumed. If the spread is a useful uncertainty
estimate, then points where the members disagree should be points where the ensemble mean is wrong.

Three views: a scatter of absolute error against spread with a rank correlation, the mean absolute
error in bins of increasing spread, and a check of how much of the total error is concentrated in
the least confident predictions.

In [ ]:
# --- Is the spread informative? -----------------------------------------------
abs_err = np.abs(mean_pred - yd_te)
rho, pval = spearmanr(spread, abs_err)

nb_bin = 5
order_s = np.argsort(spread)
bins = np.array_split(order_s, nb_bin)
bin_sp  = [spread[b].mean()  for b in bins]
bin_err = [abs_err[b].mean() for b in bins]
bin_r2  = [r2_score(yd_te[b], mean_pred[b]) for b in bins]

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(14, 4.1))

a1.scatter(spread, abs_err, s=12, alpha=0.45, color=C_DATA)
b = np.polyfit(spread, abs_err, 1)
xs = np.linspace(spread.min(), spread.max(), 50)
a1.plot(xs, np.polyval(b, xs), color=C_FIT, lw=2)
a1.set_xlabel("ensemble spread (GPa)"); a1.set_ylabel("|error| of ensemble mean (GPa)")
a1.set_title(f"Spearman rank correlation {rho:+.3f}\n(p = {pval:.1e})", fontsize=10)

a2.bar(np.arange(nb_bin), bin_err, color=C_DATA, width=0.6)
a2.plot(np.arange(nb_bin), bin_sp, "-o", color=C_FIT, lw=2, ms=5, label="mean spread")
a2.set_xticks(np.arange(nb_bin))
a2.set_xticklabels([f"{i+1}" for i in range(nb_bin)])
a2.set_xlabel(f"quintile of spread, least to most uncertain")
a2.set_ylabel("GPa"); a2.legend(fontsize=8)
a2.set_title("mean |error| by spread quintile", fontsize=10)

a3.bar(np.arange(nb_bin), bin_r2, color=[C_ALT if r > 0 else C_BAD for r in bin_r2], width=0.6)
a3.axhline(0, color="k", lw=0.8)
a3.set_xticks(np.arange(nb_bin)); a3.set_xticklabels([f"{i+1}" for i in range(nb_bin)])
a3.set_xlabel("quintile of spread"); a3.set_ylabel("test $R^2$ within the quintile")
a3.set_title("accuracy within each confidence band", fontsize=10)
plt.tight_layout(); plt.show()

print(f"{'quintile':>9s} {'mean spread':>12s} {'mean |error|':>13s} {'R2 in bin':>11s}")
for i in range(nb_bin):
    print(f"{i+1:>9d} {bin_sp[i]:>12.3f} {bin_err[i]:>13.3f} {bin_r2[i]:>11.3f}")
print(f"\nmean |error| in the most confident fifth : {bin_err[0]:.3f} GPa")
print(f"mean |error| in the least confident fifth: {bin_err[-1]:.3f} GPa")
print(f"ratio: {bin_err[-1]/max(bin_err[0],1e-9):.2f}x")


**What the three panels show.** Left: the absolute error of the ensemble mean against the
ensemble spread $u$, one point per test microstructure, with the Spearman rank correlation in the
title. Middle: the test set split into five equal groups ordered by spread, with the mean absolute
error as bars and the mean spread as the orange line. Right: the $R^2$ achieved within each of those
five groups.

If the spread were useless the bars in the middle panel would be flat and the right panel would be
level. The direction of those two panels, and the ratio printed underneath, are the evidence that
the ordering carries information.

### Reading the rank correlation

The Spearman coefficient printed above is the headline. A value near zero would mean the spread
carries no information about the error and should not be reported as an uncertainty. A clearly
positive value means the ordering is useful: the model's least confident predictions really are its
worst, and that is enough to build a triage rule on even if the numbers are not calibrated
probabilities.

Be careful about what it does not say.

The spread measures disagreement between members trained on the same data with the same
architecture. It captures the part of the uncertainty that comes from the optimisation and the
initialisation. It does **not** capture a bias shared by every member, and every member here shares
the training set, the architecture and the augmentation. Systematic error is invisible to it, which
is precisely why the ensemble in Part 4's situation would have been confidently wrong.

An ensemble standard deviation is also not a calibrated error bar. Checking calibration needs a
coverage plot: what fraction of test points fall inside one, two and three spreads.

### Interactive: using the spread as a triage rule

The practical use is selective prediction. Accept the model's answer where the spread is small,
send the rest for a full FE run. The slider sets the fraction sent for simulation and the panel
reports the accuracy on what is left.

In [ ]:
# --- Interactive: selective prediction ---------------------------------------
def triage(reject_percent=20):
    keep_n = int(round(len(spread) * (1 - reject_percent / 100.0)))
    keep_n = max(20, keep_n)
    keep = np.argsort(spread)[:keep_n]
    r2_keep = r2_score(yd_te[keep], mean_pred[keep])
    thr = spread[np.argsort(spread)][keep_n - 1]

    fig, ax = plt.subplots(figsize=(5.6, 4.6))
    lim = [yd_te.min() - 0.25, yd_te.max() + 0.25]
    mask = np.ones(len(spread), bool); mask[keep] = False
    ax.plot(lim, lim, "k--", lw=1)
    ax.scatter(yd_te[keep], mean_pred[keep], s=14, alpha=0.6, color=C_DATA, label="accepted")
    ax.scatter(yd_te[mask], mean_pred[mask], s=18, alpha=0.7, color=C_BAD,
               label="sent to FE")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("true $\\Delta E$ (GPa)"); ax.set_ylabel("ensemble mean (GPa)")
    ax.set_title(f"reject {reject_percent}% least confident\n"
                 f"$R^2$ on the accepted {keep_n} of {len(spread)}: {r2_keep:+.3f}", fontsize=10)
    ax.legend(fontsize=8, loc="upper left")
    plt.tight_layout(); plt.show()
    print(f"spread threshold {thr:.3f} GPa")
    print(f"R2 on everything      {R2_ENS:+.4f}")
    print(f"R2 on the accepted    {r2_keep:+.4f}   ({r2_keep - R2_ENS:+.4f})")

interact(triage, reject_percent=IntSlider(20, min=0, max=60, step=5, continuous_update=False));


---

# What to take away

1. **Measure the learning curve first, and measure it with more than one seed.** Training the same
   model on nested subsets costs a fraction of a proper tuning run and tells you whether the next
   gain comes from more FE data or from a better model. Part 1 measured the seed-to-seed spread at a
   fixed training size alongside the change from one size to the next, because a single run per point
   cannot separate the two.
2. **Augmentation is a claim about symmetry.** Periodic roll is exact here because the cells are
   periodic. Mirrors preserve $E_{22}$ and $E_{33}$ separately. A 90 degree rotation exchanges them,
   so it is valid only with the target transformed, and applying it without transforming the target
   drives the prediction towards zero.
3. **Know what your bound actually bounds.** Voigt and Reuss bound the stiffness and compliance
   tensors. The Reuss harmonic mean is then a rigorous lower bound on Young's modulus; the rule of
   mixtures is the uniform strain estimate and Part 3 exhibited exact composites that exceed it. What
   the notebook imposed was an envelope fitted to data, not a theorem.
4. **Bounds can be built in, but check what the gain really came from.** Squashing the output into a
   fitted envelope with a logistic makes an out-of-envelope prediction impossible by construction.
   Part 3 separated that from the output offset the same head supplies, and most of the apparent gain
   was the offset. The value of the constraint is in what it prevents, not in what it adds.
5. **A random split measures interpolation and nothing else.** Splitting by volume fraction instead
   showed the tree model saturating and the CNN degrading badly, while the two-parameter model in
   $v$ and $v^2$ held up best despite being the weakest inside the range.
6. **An envelope fitted to data holds only where the data were.** Refitted on 10 to 40 percent, it
   excluded real microstructures at 60 percent.
7. **An ensemble of four gives a usable uncertainty for four times the cost.** The spread ranks the
   errors, which is enough for a triage rule, but it sees only the variance between members. It is
   blind to any bias they share, including the one that ruins the extrapolation in Part 4.

---

# Exercises

### 1. Extend the learning curve
Add points at 5 percent and 75 percent of the training set and redraw the curve, keeping three seeds
per point. Then try to fit a power law to $1 - R^2$ against $n$. State how many of your points you
had to discard to do it, what the fitted exponent was, and whether you would put that exponent in a
paper. Part 1 printed the spread you need to answer the last part.

### 2. Make the augmentation sweep trustworthy
Each bar in Part 2 is one seed, and Part 1 measured how far a single seed can move. Repeat the `none`
and `roll+flip` runs with three seeds each, report the mean and the range, and state whether the gap
you saw survives. If it does not, say so.

### 3. Rotation on the other target
$E_{mean}$ is unchanged by a 90 degree rotation, so the rotation can be used on that target with no
correction at all. Retrain the $E_{mean}$ model with `aug="roll+flip+rot"` and `target_parity="even"`
and report whether it helped. Explain why the same transformation needs different treatment for the
two targets.

### 4. A tighter bound
Replace the fitted envelope with a purely statistical one: the 1st and 99th percentile of $E_{mean}$
within each volume fraction group of the training set, interpolated in $v$. Rebuild the bounded model
with it and report the change in $R^2$ against the standardised unbounded baseline, and the fraction
of that baseline's predictions the new envelope would have clipped. Then state the price: both
envelopes are fitted, so what would have to be true of the data for either to be trusted outside the
range it was fitted on?

### 5. The simple model won
Take the extrapolation results from Part 4 and write the two sentences you would put in a report to
a colleague who wants to use one of these surrogates to screen designs at 55 percent volume
fraction. Name which model you would give them and why, including the accuracy you are giving up
inside the training range to get it.

### 6. Uncertainty where it matters
Run the Part 5 ensemble again, but train the members on the volume fraction 10 to 40 percent split
from Part 4, and compare the ensemble spread at 50 and 60 percent with the spread inside the
training range. Does the spread grow enough to warn a user that the model is extrapolating? Relate
your answer to point 6 in the takeaways, and say what an ensemble would have to vary, beyond the
random seed, for the warning to be reliable.
